## Setup

In [153]:
import sys
import os
import importlib

from tabulate import tabulate
import matplotlib.pyplot as plt

sys.path.append("../src")

import utils
import plot
import rstats

# Reload modules to apply any changes
importlib.reload(utils)
importlib.reload(plot)
importlib.reload(rstats)

<module 'rstats' from '/Users/jsn/dev/nlp-semantic/notebooks/../src/rstats.py'>

In [154]:
FILENAME = os.getenv("FILENAME", "UFABC_PLT_combined")
BACKEND = os.getenv("BACKEND", "gemini")
DST = f"../results/{BACKEND}/{FILENAME}/"

print(f"{FILENAME=}")
print(f"{BACKEND=}")

df = utils.load(f"../results/{BACKEND}/{FILENAME}/metrics.csv")
print(f"{df.shape=}")

FILENAME='Parkinson_paper'
BACKEND='qwen'
df.shape=(14705, 13)


In [155]:
tmp = ["id", "category", "concept"] if "category" in df.columns else ["id", "concept"]
grouped = df.groupby(tmp, as_index=False)
dfx = grouped.mean(numeric_only=True)
print(tabulate(dfx.head(3), headers="keys", showindex=False))

id             category    concept       num    distance_next    entropy    distance_centroid_order    distance_centroid_static          MDS1          MDS2    vel_magnitude    acc_magnitude
-------------  ----------  ---------  ------  ---------------  ---------  -------------------------  --------------------------  ------------  ------------  ---------------  ---------------
AR_F17_CN_102  CN          ARBOL      8788         0.00682643  0.0277778                  0.0414659                    0.107898  -1.06896e-17  -1.57532e-17          5.77061          7.84125
AR_F17_CN_102  CN          AVION      8955.5       0.0058376   0.020402                   0.0413935                    0.109424  -2.42861e-17  -2.33841e-17          4.94214          6.20503
AR_F17_CN_102  CN          CAMA       8752         0.012866    0.0294118                  0.0506007                    0.105665   2.73591e-17   3.96508e-19          7.5023           9.42241


## Analysis

In [156]:
figsize = {
    "UFABC_PLT_combined": (6, 8),
    "Swear_fluency": (6, 10),
    "CPN120": (4, 8),
    "Dados_Italian_2": (7, 7),
    "German_data": (8, 8),
    "Parkinson_paper": (4, 6),
}

ylabels = {
    "acc_magnitude": "Acceleration",
    "vel_magnitude": "Velocity",
    "distance_centroid_static": "Distance to centroid",
    "distance_next": "Distance to next point",
    "entropy": "Entropy",
}

xlabels = {
    "UFABC_PLT_combined": {
        "ABSTRACT_CONCEPT": "Abstract Concept",
        "ABSTRACT_VERB": "Abstract Verb",
        "CONCRETE_CONCEPT": "Concrete Concept",
        "CONCRETE_VERB": "Concrete Verb",
    },
    "Swear_fluency": {
        "ANIMAL": "Animal",
        "A_LETTER": "Letter A",
        "F_LETTER": "Letter F",
        "S_LETTER": "Letter S",
        "SWEAR_WORDS": "Swear Words",
    },
    "CPN120": {
        "ABSTRACT": "Abstract",
        "CONCRETE": "Concrete",
    },
    "Dados_Italian_2": {
        "bird": "Bird",
        "bodypart": "Body Part",
        "building": "Building",
        "clothing": "Clothing",
        "fruit": "Fruit",
        "furniture": "Furniture",
        "implement": "Implement",
        "mammal": "Mammal",
        "vegetable": "Vegetable",
    },
    "German_data": {
        "bird": "Bird",
        "bodypart": "Body Part",
        "building": "Building",
        "clothing": "Clothing",
        "fruit": "Fruit",
        "furniture": "Furniture",
        "implement": "Implement",
        "mammal": "Mammal",
        "vegetable": "Vegetable",
    },
    "Parkinson_paper": {
        "CN": "HC",
        "DF": "bvFTD",
        "PD": "PD",
    },
}

import pandas as pd


# TODO: review actual p-value thresholds (from statsannotations?)
def stars(p):
    if pd.isna(p):
        return ""
    if p < 1e-3:
        return "***"
    if p < 1e-2:
        return "**"
    if p < 5e-2:
        return "*"
    return ""

In [180]:
import io
import contextlib
import pandas as pd
import numpy as np
import seaborn as sns

metrics = [
    "entropy",
    "distance_next",
    "distance_centroid_static",
    "vel_magnitude",
    "acc_magnitude",
]

if "category" not in dfx.columns:
    dfx = dfx.rename(columns={"concept": "category"})

cats = list(dfx["category"].unique())
fig, axes = plt.subplots(2, len(metrics), figsize=(4 * len(metrics), 7))

for i, metric in enumerate(metrics):
    print(f"[{i + 1}/{len(metrics)}] Analyzing '{metric}'")

    # Use lognormal for all metrics, except for distance_centroid_static
    family = "lognormal" if metric != "distance_centroid_static" else "gaussian"

    stdout = io.StringIO()
    with contextlib.redirect_stdout(stdout):
        # Fit GLMM and get emmeans + Tukey pairs
        glmm = rstats.glmm(df, formula=f"{metric} ~ category + (1|id)", family=family)
        pred = rstats.emmeans(effect="category")
        pairs = rstats.pairs()

    # Individual figure
    ax1 = plot.boxplot(dfx, metric, pred, pairs, ax=None, figsize=figsize.get(FILENAME))
    ax1.set_title(ylabels.get(metric, metric))
    plt.tight_layout()
    plt.savefig(f"{DST}/boxplot-{metric}.png", bbox_inches="tight")
    plt.close()

    # Combined figure
    ax2 = plot.boxplot(dfx, metric, pred, pairs, ax=axes[0, i])
    ax2.set_title(ylabels.get(metric, metric))

    with open(f"{DST}/r-output-{metric}.txt", "w") as fp:
        fp.write(stdout.getvalue())

    # Build a (symmetric) matrix of p-values
    pmat = pd.DataFrame(1.0, index=cats, columns=cats, dtype=float)
    for _, row in pairs.iterrows():
        g1, g2 = str(row["group1"]), str(row["group2"])
        p = float(row["p.value"])
        pmat.loc[g1, g2] = p
        pmat.loc[g2, g1] = p

    # TODO: draw heatmaps with a shared color scale
    np.fill_diagonal(pmat.values, np.nan)
    pmat.to_csv(f"{DST}/pvalues-{metric}.csv", float_format="%.3g")
    mask = np.tril(np.ones_like(pmat, dtype=bool))

    ax3 = axes[1, i]
    ax3.grid(False)
    hm = sns.heatmap(
        pmat.astype(float),
        mask=mask,
        # annot=pmat.map(stars).to_numpy(),
        cmap=sns.light_palette("seagreen", as_cmap=True, reverse=True),
        cbar=False,
        # TODO: are values always between 0 and 1?
        vmin=0,
        vmax=1,
        xticklabels=cats,
        yticklabels=cats,
        ax=ax3,
        square=True,
    )
    
    ax3.tick_params(axis="x", rotation=90)
    # ax3.xaxis.set_ticks_position("top")
    # ax3.xaxis.set_label_position("top")
    # ax3.tick_params(axis="both", which="both", length=0)
    # for i, label in enumerate(pmat.index):
    #     ax3.text(i + 0.5, i + 0.5, label, ha="right", va="center")

    ann = pmat.map(stars)
    for r, row_lab in enumerate(cats):
        for c, col_lab in enumerate(cats):
            if r >= c:
                continue

            s = stars(pmat.iat[r, c])
            if not s:
                continue

            ax3.text(
                c + 0.5,
                r + 0.5,
                s,
                ha="center",
                va="center",
                fontsize=20,
                fontweight="bold",
                color="black",
            )

    # save raw data
    pmat.to_csv(f"{DST}/pvals-{metric}.csv", float_format="%.3g")

    # plt.tight_layout()
    # fig_h.savefig(f"{DST}/heatmap_{metric}.png", bbox_inches="tight", dpi=300)
    # plt.close(fig_h)

cbar_ax = fig.add_axes([0.25, -0.05, 0.5, 0.03])
fig.colorbar(hm.collections[0], cax=cbar_ax, orientation="horizontal")

plt.tight_layout()
fig.savefig(f"{DST}/main.png", bbox_inches="tight")
plt.close()

[1/5] Analyzing 'entropy'
[2/5] Analyzing 'distance_next'
[3/5] Analyzing 'distance_centroid_static'
[4/5] Analyzing 'vel_magnitude'
[5/5] Analyzing 'acc_magnitude'


/var/folders/2t/jw0zh_ts0q19czmdc_ssfz2h0000gn/T/ipykernel_31084/2031234785.py:116: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
